# Trabalho Prático - Notebook do grupo

**Inteligência Artificial - Graduação** · Prof. Bruno Nogueira · FACOM/UFMS · 2026/2

**Grupo:** _preencha_ · **Tema:** _preencha_

---

Este é o notebook **entregável** do grupo. Ele precisa rodar do início ao fim
("Kernel → Restart & Run All") e **regerar os modelos** salvos em `modelos/`. A
estrutura segue o `00_exemplo_biblioteca.ipynb`; consulte-o sempre que tiver
dúvida.

Marcadores usados aqui:

* `🔨 TODO` - algo que o grupo precisa fazer/decidir;
* `📝 RELATÓRIO` - um resultado que deve aparecer (e ser discutido) no artigo.

Requisitos mínimos de cada seção estão no enunciado. Pode acrescentar seções,
células e análises à vontade - só não remova as seções numeradas.

---
## 0. Ambiente

In [5]:
import os
import subprocess
import sys
from pathlib import Path

# 🔨 TODO (só para o Colab): endereço do repositório do grupo
REPO_URL = "https://github.com/GabrielSakiyama/IA-2026-02-TP-Chatbot"

if "google.colab" in sys.modules:
    pasta = Path("/content") / Path(REPO_URL).stem
    if not pasta.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(pasta)], check=True)
    os.chdir(pasta)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

raiz = Path.cwd()
while not (raiz / "chatbot").is_dir():
    if raiz.parent == raiz:
        raise RuntimeError("Rode este notebook de dentro do repositório do trabalho.")
    raiz = raiz.parent
os.chdir(raiz)
sys.path.insert(0, str(raiz))

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

from chatbot.avaliacao import avaliar, tabela_resumo
from chatbot.config import carregar_config
from chatbot.dados import adicionar_escopo, carregar_dados, resumo, validar_dados, vazamento
from chatbot.grafo import Conversa, construir_grafo
from chatbot.modelos import carregar_classificadores, salvar_classificadores
from chatbot.representacoes import tfidf_caracteres, tfidf_palavras
from chatbot.rotulos import CLASSES_ESCOPO

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CORES = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]
pd.set_option("display.max_colwidth", 80)

---
## 1. Configuração

🔨 TODO: preencha o `grupo.yaml` da raiz do repositório (tema e intenções
**validados com o professor**) antes de rodar esta seção.

In [ ]:
config = carregar_config("grupo.yaml")
print("Grupo:", config.grupo, "| Tema:", config.tema)
pd.DataFrame([(i.nome, i.descricao) for i in config.intencoes.values()], columns=["intenção", "descrição"])

---
## 2. Dados

🔨 TODO: descreva aqui (em markdown) **como** os dados foram construídos: quem
escreveu, que fontes foram coletadas, que LLM e que *prompts* foram usados, como os
negativos difíceis foram pensados, como o teste foi obtido (ex.: formulário
respondido por N colegas).

📝 RELATÓRIO: tabela intenção × origem do treino e do teste.

In [ ]:
treino = adicionar_escopo(carregar_dados(config.treino), config)
teste = adicionar_escopo(carregar_dados(config.teste), config)
print("Problemas no treino:", validar_dados(treino, config) or "nenhum")
print("Problemas no teste: ", validar_dados(teste, config) or "nenhum")
print("Vazamento treino→teste:", vazamento(treino, teste) or "nenhum")
display(resumo(treino))
display(resumo(teste))

### Gerando frases de treino com o LLM (opcional)

Rode no terminal (ou descomente a célula abaixo). As frases vão para
`dados/gerados_llm.csv` e os prompts ficam registrados em `dados/prompts_llm.md`.
**Revise tudo** antes de copiar para o `treino.csv`; nunca use no teste; declare no
`USO_DE_IA.md`.

In [ ]:
# Para rodar daqui, tire o "#" das linhas abaixo (gasta créditos da chave do grupo):
# subprocess.run([sys.executable, "scripts/gerar_dados_llm.py", "--n", "10"], check=True)
# gerados = pd.read_csv("dados/gerados_llm.csv")
# gerados.sample(15)

🔨 TODO: análise exploratória - tamanho das mensagens por classe, palavras mais
frequentes por intenção, exemplos de negativos difíceis, diferenças entre frases
humanas e de LLM...

In [ ]:
from chatbot.texto import normalizar

# 1. tamanho médio das mensagens (em palavras), por escopo e por origem
palavras = treino["texto"].str.split().str.len()
display(treino.assign(palavras=palavras)
        .pivot_table(index="escopo", columns="origem", values="palavras", aggfunc="mean").round(1))

# 2. termos que mais caracterizam cada rótulo: maior TF-IDF médio dentro do rótulo
vet = tfidf_palavras(ngram_range=(1, 1))
X = vet.fit_transform(treino["texto"])
termos = np.array(vet.get_feature_names_out())
destaque = {}
for rotulo in config.nomes_intencoes + ["chitchat", "fora_dominio"]:
    media = np.asarray(X[(treino["intencao"] == rotulo).to_numpy()].mean(axis=0)).ravel()
    destaque[rotulo] = ", ".join(termos[media.argsort()[::-1][:6]])
display(pd.Series(destaque, name="termos com maior TF-IDF médio").to_frame())

# 3. negativos difíceis: frases fora_dominio com palavras típicas do domínio
ISCAS = {"tcc", "prazo", "orientador", "banca", "defesa", "relatorio", "historico",
         "deposito", "depositar", "formatacao", "modelo", "normas", "abnt", "versao"}
eh_dificil = treino["texto"].map(lambda t: bool(set(normalizar(t).split()) & ISCAS))
fora = treino["intencao"] == "fora_dominio"
print(f"{(fora & eh_dificil).sum()} de {fora.sum()} frases fora_dominio usam palavras típicas do domínio")
display(treino[fora & eh_dificil].groupby("origem").size().rename("negativos difíceis").to_frame())
display(treino[fora & eh_dificil].sample(8, random_state=RANDOM_STATE)[["texto", "origem"]])

---
## 3. Representações de texto

🔨 TODO: escolha **pelo menos duas** representações para comparar. Os construtores
de `chatbot/representacoes.py` aceitam os parâmetros do `TfidfVectorizer`
(`ngram_range`, `min_df`, `stop_words`, ...). Embeddings pré-treinados
(`EmbeddingsSentenca`) são opcionais.

In [ ]:
# palavras vazias, já sem acento (o tfidf_palavras normaliza o texto antes de separar
# as palavras). As palavras de pergunta (quando, onde, como, qual) ficam de fora da
# lista de propósito: na análise exploratória, "quando" foi um dos termos mais
# típicos de prazos.
PALAVRAS_VAZIAS = [
    "a", "o", "as", "os", "um", "uma", "uns", "umas", "de", "do", "da", "dos", "das",
    "em", "no", "na", "nos", "nas", "num", "numa", "por", "pelo", "pela", "pelos", "pelas",
    "pra", "pro", "para", "com", "sem", "ao", "aos", "e", "ou", "mas", "que", "se",
    "eu", "me", "mim", "meu", "minha", "meus", "minhas", "voce", "vc", "te", "seu", "sua",
    "seus", "suas", "ele", "ela", "eles", "elas", "isso", "isto", "esse", "essa", "este",
    "esta", "ta", "to", "ja", "tb", "tbm", "la", "aqui", "ai", "ne",
]

REPRESENTACOES = {
    "TF-IDF palavras": tfidf_palavras,
    "TF-IDF palavras sem vazias": lambda: tfidf_palavras(stop_words=PALAVRAS_VAZIAS),
    "TF-IDF caracteres": tfidf_caracteres,
}

---
## 4. Classificador de escopo (dominio / chitchat / fora_dominio)

🔨 TODO: compare **pelo menos três algoritmos vistos em aula**, com validação
cruzada estratificada **só no treino**. Classificadores que ainda serão
apresentados na disciplina (KNN, Naive Bayes, SVM...) podem entrar quando forem
vistos.

📝 RELATÓRIO: tabela/gráfico da comparação, com média e desvio entre folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

CLASSIFICADORES = {
    "Regressão logística": lambda: LogisticRegression(C=10, max_iter=2000),
    "Rede neural (MLP)": lambda: MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=600,
                                               random_state=RANDOM_STATE),
    "Árvore de decisão": lambda: DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN (k=5)": lambda: KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": lambda: MultinomialNB(),
}


def comparar(textos, rotulos):
    linhas = []
    for nome_rep, rep in REPRESENTACOES.items():
        for nome_clf, clf in CLASSIFICADORES.items():
            pipe = Pipeline([("vetorizador", rep()), ("classificador", clf())])
            r = cross_validate(pipe, textos, rotulos, cv=cv, scoring=["f1_macro", "accuracy"],
                               return_train_score=True, n_jobs=-1)
            linhas.append({"representação": nome_rep, "classificador": nome_clf,
                           "F1 macro (CV)": r["test_f1_macro"].mean(), "desvio": r["test_f1_macro"].std(),
                           "acurácia (CV)": r["test_accuracy"].mean(),
                           "F1 macro (treino)": r["train_f1_macro"].mean()})
    return pd.DataFrame(linhas).sort_values("F1 macro (CV)", ascending=False).reset_index(drop=True)


tab_escopo = comparar(treino["texto"], treino["escopo"])
tab_escopo.round(3)

**Escolha do modelo de escopo.** Na comparação acima, a rede neural (MLP) e a
regressão logística, ambas com n-gramas de caracteres, ficaram empatadas: a
diferença entre as médias foi menor que o desvio entre os folds. O grupo escolheu a
**regressão logística**, por ser mais simples, mais rápida e interpretável (dá para
ver o peso de cada n-grama). Abaixo, o ajuste de hiperparâmetros com `GridSearchCV`
e a curva de validação do C.

In [ ]:
busca_escopo = GridSearchCV(
    Pipeline([("vetorizador", tfidf_caracteres()), ("classificador", LogisticRegression(max_iter=2000))]),
    param_grid={"classificador__C": [0.01, 0.1, 1, 10, 100, 1000],     # C = 1/lambda
                "classificador__class_weight": [None, "balanced"],     # balanced: classes menores pesam mais
                "vetorizador__ngram_range": [(2, 4), (2, 5)]},
    scoring="f1_macro", cv=cv, n_jobs=-1, return_train_score=True,
)
busca_escopo.fit(treino["texto"], treino["escopo"])
clf_escopo = busca_escopo.best_estimator_
print(busca_escopo.best_params_, round(busca_escopo.best_score_, 3))

In [ ]:
def curva_validacao(busca, titulo):
    """F1 de treino e de validação para cada C, com os outros hiperparâmetros nos melhores valores."""
    res = pd.DataFrame(busca.cv_results_)
    for nome, valor in busca.best_params_.items():
        if nome != "classificador__C":
            res = res[res[f"param_{nome}"].astype(str) == str(valor)]
    res = res.sort_values("param_classificador__C")
    C = res["param_classificador__C"].astype(float)

    fig, ax = plt.subplots(figsize=(6, 3.5))
    for parte, nome, cor in [("train", "treino", CORES[0]), ("test", "validação (CV)", CORES[1])]:
        media, desvio = res[f"mean_{parte}_score"], res[f"std_{parte}_score"]
        ax.plot(C, media, "o-", color=cor, label=nome)
        ax.fill_between(C, media - desvio, media + desvio, color=cor, alpha=0.15)
    ax.set_xscale("log")
    ax.set_xlabel("C (= 1/λ; maior = menos regularização)")
    ax.set_ylabel("F1 macro")
    ax.set_title(titulo)
    ax.legend()
    plt.show()
    return res[["param_classificador__C", "mean_train_score", "mean_test_score", "std_test_score"]].round(3)


curva_validacao(busca_escopo, "Escopo: curva de validação")

📝 RELATÓRIO: matriz de confusão (validação cruzada), taxa de falso aceite e de
falsa recusa, e análise dos erros.

**Qual erro é mais grave no nosso domínio?** O **falso aceite**. Quando o bot
responde com confiança uma pergunta de outro assunto (por exemplo, o prazo do
estágio respondido com as regras do TCC), ele confunde o estudante e pode levá-lo a
perder um prazo de verdade. Na falsa recusa, o estudante é encaminhado à Coordenação
ou à FACOM e não recebe informação errada.

In [ ]:
prev_cv = cross_val_predict(clf_escopo, treino["texto"], treino["escopo"], cv=cv)
ConfusionMatrixDisplay.from_predictions(treino["escopo"], prev_cv, labels=CLASSES_ESCOPO,
                                        cmap="Blues", colorbar=False)
plt.grid(False)
plt.show()

real = treino["escopo"].to_numpy()
fora_do_tema = real != "dominio"
print(f"falso aceite: {(fora_do_tema & (prev_cv == 'dominio')).sum() / fora_do_tema.sum():.1%} "
      "das mensagens chitchat/fora_dominio foram tratadas como do domínio")
print(f"falsa recusa: {(~fora_do_tema & (prev_cv != 'dominio')).sum() / (~fora_do_tema).sum():.1%} "
      "das mensagens do domínio foram recusadas ou tratadas como conversa")

# os erros se concentram em alguma origem?
erros = treino.assign(previsto=prev_cv).query("escopo != previsto")
display(pd.DataFrame({"frases": treino["origem"].value_counts(), "erros": erros["origem"].value_counts()})
        .fillna(0).astype(int).assign(taxa=lambda d: (d["erros"] / d["frases"]).round(3)))
erros[["texto", "escopo", "previsto", "origem"]]

---
## 5. Classificador de tema (intenções do domínio)

🔨 TODO: mesma metodologia, treinando **só** com as mensagens do domínio.

In [ ]:
treino_dom = treino[treino["escopo"] == "dominio"]
tab_tema = comparar(treino_dom["texto"], treino_dom["intencao"])
tab_tema.round(3)

**Escolha do modelo de tema.** A regressão logística com n-gramas de caracteres
ficou entre as melhores, empatada com a MLP (caracteres) e com a regressão
logística de palavras sem palavras vazias. O grupo a escolheu por ser o mesmo modelo
do escopo (mais simples de manter e explicar) e porque o limiar de confiança
(seção 6) depende das probabilidades, que na regressão logística costumam ser mais
confiáveis que as do Naive Bayes.

Repare que, no tema, tirar as palavras vazias ajudou, ao contrário do escopo: aqui
todas as frases são de estudantes falando do TCC, então "meu", "pra" e "como"
aparecem em todas as intenções e não distinguem nenhuma.

In [ ]:
# as 6 intenções têm o mesmo número de frases, então não precisa de class_weight
busca_tema = GridSearchCV(
    Pipeline([("vetorizador", tfidf_caracteres()), ("classificador", LogisticRegression(max_iter=2000))]),
    param_grid={"classificador__C": [0.01, 0.1, 1, 10, 100, 1000],
                "vetorizador__ngram_range": [(2, 4), (2, 5)]},
    scoring="f1_macro", cv=cv, n_jobs=-1, return_train_score=True,
)
busca_tema.fit(treino_dom["texto"], treino_dom["intencao"])
clf_tema = busca_tema.best_estimator_
print(busca_tema.best_params_, round(busca_tema.best_score_, 3))

In [ ]:
curva_validacao(busca_tema, "Tema: curva de validação")

In [ ]:
prev_tema = cross_val_predict(clf_tema, treino_dom["texto"], treino_dom["intencao"], cv=cv)
ConfusionMatrixDisplay.from_predictions(treino_dom["intencao"], prev_tema, labels=config.nomes_intencoes,
                                        cmap="Blues", colorbar=False, xticks_rotation=45)
plt.grid(False)
plt.show()
print(classification_report(treino_dom["intencao"], prev_tema, zero_division=0))

# quais intenções mais se confundem (verdadeira -> prevista)?
erros_tema = treino_dom.assign(previsto=prev_tema).query("intencao != previsto")
display(erros_tema.groupby(["intencao", "previsto"]).size().sort_values(ascending=False)
        .rename("erros").to_frame())

# os erros se concentram em alguma origem?
display(pd.DataFrame({"frases": treino_dom["origem"].value_counts(), "erros": erros_tema["origem"].value_counts()})
        .fillna(0).astype(int).assign(taxa=lambda d: (d["erros"] / d["frases"]).round(3)))
erros_tema[["texto", "intencao", "previsto", "origem"]]

---
## 6. Limiar de confiança

🔨 TODO: escolha `roteamento.limiar_tema` (no `grupo.yaml`) com base nas
probabilidades da validação cruzada, como na seção 9 do exemplo. Justifique.

In [ ]:
def confianca_cv(modelo):
    """Maior probabilidade de cada frase na validação cruzada e se a previsão acertou."""
    proba = cross_val_predict(modelo, treino_dom["texto"], treino_dom["intencao"], cv=cv,
                              method="predict_proba")
    classes = np.array(sorted(treino_dom["intencao"].unique()))
    return proba.max(axis=1), classes[proba.argmax(axis=1)] == treino_dom["intencao"].to_numpy()


def tabela_limiar(confianca, acertou, limiares=(0.0, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7)):
    linhas = []
    for limiar in limiares:
        respondidas = confianca >= limiar
        linhas.append({"limiar": limiar,
                       "% respondidas direto": respondidas.mean(),
                       "acurácia nas respondidas": acertou[respondidas].mean() if respondidas.any() else np.nan,
                       "erros evitados": int((~acertou & ~respondidas).sum()),
                       "acertos que viraram pergunta": int((acertou & ~respondidas).sum())})
    return pd.DataFrame(linhas).round(3)


# C = 10 e C = 1000 empataram no F1 (curva de validação do tema); a diferença aparece nas probabilidades
ngram = busca_tema.best_params_["vetorizador__ngram_range"]
fig, eixos = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
bins = np.linspace(0, 1, 21)
tabelas = {}
for ax, C in zip(eixos, (10, 1000)):
    modelo = Pipeline([("vetorizador", tfidf_caracteres(ngram_range=ngram)),
                       ("classificador", LogisticRegression(C=C, max_iter=2000))])
    confianca, acertou = confianca_cv(modelo)
    tabelas[C] = tabela_limiar(confianca, acertou)
    ax.hist(confianca[acertou], bins=bins, alpha=0.7, color=CORES[2], label=f"acertos ({acertou.sum()})")
    ax.hist(confianca[~acertou], bins=bins, alpha=0.8, color=CORES[3], label=f"erros ({(~acertou).sum()})")
    ax.set_title(f"Tema: confiança das previsões (CV), C = {C}")
    ax.set_xlabel("maior probabilidade do classificador de tema")
    ax.legend()
eixos[0].set_ylabel("frases")
plt.show()
for C, tab in tabelas.items():
    print(f"C = {C}")
    display(tab)

**Escolha do C e do limiar.** No F1, C = 10 e C = 1000 empataram, mas as
probabilidades são bem diferentes. Sem regularização (C = 1000), o modelo fica
confiante até nos erros, e o limiar quase não os separa dos acertos. Com C = 10, os
erros se concentram nas confianças baixas. O grupo escolheu **C = 10** e limiar
**0,35** (gravado em `roteamento.limiar_tema`, no `grupo.yaml`): na validação
cruzada, esse limiar evita 11 dos 24 erros e transforma só 6 acertos em perguntas.

Dois cuidados: na avaliação automática, a rota "esclarecer" conta como erro, então
o limiar só custa pontos ali; e essas confianças vêm de um treino majoritariamente
gerado por LLM, então no teste (escrito por pessoas) o bot deve perguntar mais do que
a tabela indica. O grupo considerou que perguntar é melhor que responder errado com
confiança (mesmo critério usado no escopo).

In [ ]:
# modelo de tema final (C = 10); é este que a seção 7 salva
clf_tema = Pipeline([("vetorizador", tfidf_caracteres(ngram_range=ngram)),
                     ("classificador", LogisticRegression(C=10, max_iter=2000))])
f1 = cross_validate(clf_tema, treino_dom["texto"], treino_dom["intencao"], cv=cv, scoring="f1_macro")["test_score"]
print(f"tema com C = 10: F1 macro (CV) = {f1.mean():.3f} ± {f1.std():.3f}")
clf_tema.fit(treino_dom["texto"], treino_dom["intencao"])
print("limiar no grupo.yaml:", carregar_config("grupo.yaml").limiar_tema)

---
## 7. Salvar os modelos finais

In [ ]:
salvar_classificadores(clf_escopo, clf_tema, config)
config = carregar_config("grupo.yaml")  # relê, caso o limiar tenha sido alterado

---
## 8. O chatbot funcionando (com o LLM do OpenRouter)

🔨 TODO: reúna os documentos em `documentos/` (uma subpasta por intenção + `geral/`,
com o `FONTES.md`), ajuste `persona` e `contato` no `grupo.yaml` e mostre conversas
que ilustrem acertos **e** limitações do bot (inclusive perguntas cuja resposta
não está nos documentos). Se o grupo alterou o grafo (`chatbot/grafo.py`),
explique aqui o que mudou.

Antes: configure a chave no `.env` (ou nos *Secrets* do Colab) e rode
`python scripts/testar_llm.py`.

In [ ]:
from chatbot.conhecimento import BaseConhecimento
from chatbot.llm import ClienteLLM, ErroLLM, prompt_intencao

base = BaseConhecimento.carregar_ou_criar(config)   # cria o índice na 1ª vez
display(base.resumo())

# teste da busca (não gasta créditos): os trechos vêm só da pasta da intenção e de geral/
pergunta, intencao = "posso fazer tcc em dupla?", "inicio_tcc"
for tr, sim in base.buscar(pergunta, intencao=intencao):
    print(f"{sim:.2f}  {tr.fonte} | {tr.texto[:80]!r}")

In [ ]:
# o prompt que o LLM recebe para essa pergunta
for m in prompt_intencao(config, intencao, pergunta, trechos=base.buscar(pergunta, intencao=intencao)):
    print(f"--- {m['role']} ---\n{m['content']}\n")

In [ ]:
try:
    llm = ClienteLLM.da_config(config)
except ErroLLM as erro:
    print("Sem LLM:", erro)
    llm = None

# acerto, pergunta sem resposta nos documentos, negativo difícil, regra da data da defesa
# (prazos), e um erro do bot (defesa de mestrado aceita como banca do TCC)
mensagens = [
    "oi!",
    "posso fazer tcc em dupla?",
    "quantos dias tenho pra entregar a versão final depois da defesa?",
    "qual prazo do relatório final do estágio?",
    "qual data da minha defesa?",
    "como funciona a defesa do mestrado?",
    "valeu, tchau",
]
conversa = Conversa(config, clf_escopo, clf_tema, llm=llm, base=base)
for m in mensagens:
    e = conversa.enviar(m)
    print(f"você: {m}\n  [rota={e['rota']} | resposta {e['fonte_resposta']}]\n  bot: {e['resposta']}\n")
if llm:
    print("Uso do LLM:", llm.uso)

---
## 9. Avaliação final no teste (uma única vez)

📝 RELATÓRIO: tabela dos três níveis, matriz de confusão ponta a ponta e análise
dos erros (separando, se possível, por origem das frases).

In [ ]:
clf_escopo, clf_tema = carregar_classificadores(config)
grafo = construir_grafo(config, clf_escopo, clf_tema)
resultado, previsoes = avaliar(config, clf_escopo, clf_tema, grafo, teste)
tab, extras = tabela_resumo(resultado)
display(tab.round(3))
display(extras.round(3))
previsoes.loc[~previsoes["acertou"], ["texto", "real", "rota", "confianca_escopo", "confianca_intencao"]]

---
## 10. Uso de IA generativa

🔨 TODO: confira se **todo** uso de IA do grupo (geração de dados, LLM do bot,
código, escrita, ideias, depuração...) está no `USO_DE_IA.md` e será copiado para a
seção própria do artigo e mencionado no vídeo. **Uso não declarado = nota zero
(plágio).**

---
## 11. Verificação da entrega

Deve terminar com **0 erro(s)**.

In [ ]:
!{sys.executable} scripts/verificar_entrega.py